# Exploratory analysis of customer feedback

This notebook is the **exploration** step. Everything that proved useful here was moved into
tested modules under `src/` - the notebook only imports them. Run `python -m src.pipeline` first
so the database exists.

Questions:
1. What does the raw data look like and how dirty is it?
2. Are star ratings a reasonable weak label for sentiment?
3. What do customers complain about, and does it change over time?
4. Which segments are least satisfied?

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
from src.data_generator import generate_feedback
from src.preprocessing import preprocess
from src.database import get_engine, read_feedback_full, run_query
from src import analytics

pd.set_option("display.max_colwidth", 90)

## 1. Raw data and data quality

In [2]:
raw = generate_feedback(n=5000)
print(raw.shape)
raw.head()

(5252, 8)


,feedback_id,created_at,channel,plan,region,customer_segment,rating,text
0,FB003433,2025-01-01 01:00:00,survey,data_only,Munich,new_customer,2,"Too expensive for so little data, competitors are cheaper Would recommend."
1,FB003429,2025-01-01 03:00:00,survey,data_only,East,loyal_customer,5,"Flexible contract, monthly cancellation is a big plus"
2,FB000823,2025-01-01 03:00:00,email,prepaid,West,new_customer,3,"It is okay, nothing special but works most of the time Keep it up!"
3,FB002832,2025-01-01 06:00:00,email,prepaid,West,loyal_customer,2,"They switched my plan without asking, very unfair contract terms Please fix this."
4,FB001437,2025-01-01 09:00:00,survey,family,Munich,new_customer,3,"Service is acceptable, price is average I expected better."


In [3]:
quality = pd.Series({
    "rows": len(raw),
    "duplicate feedback_id": raw["feedback_id"].duplicated().sum(),
    "empty text": (raw["text"].str.strip() == "").sum(),
    "missing rating": raw["rating"].isna().sum(),
})
quality

rows                     5252
duplicate feedback_id      52
empty text                 26
missing rating              0
dtype: int64

In [4]:
clean = preprocess(raw)
print(f"{len(raw) - len(clean)} rows removed during cleaning")
clean[["text", "clean_text", "rating", "rating_sentiment", "nps_group"]].head()

111 rows removed during cleaning


,text,clean_text,rating,rating_sentiment,nps_group
0,"Too expensive for so little data, competitors are cheaper Would recommend.",too expensive for so little data competitors are cheaper would recommend,2,negative,detractor
1,"Flexible contract, monthly cancellation is a big plus",flexible contract monthly cancellation is a big plus,5,positive,promoter
2,"It is okay, nothing special but works most of the time Keep it up!",it is okay nothing special but works most of the time keep it up,3,neutral,detractor
3,"They switched my plan without asking, very unfair contract terms Please fix this.",they switched my plan without asking very unfair contract terms please fix this,2,negative,detractor
4,"Service is acceptable, price is average I expected better.",service is acceptable price is average i expected better,3,neutral,detractor


## 2. Ratings as weak sentiment labels
The label distribution decides whether we need class weighting (we do: neutral is the minority).

In [5]:
clean["rating"].value_counts().sort_index().to_frame("count")

,count
rating,
1,1053
2,851
3,764
4,968
5,1505


In [6]:
clean["rating_sentiment"].value_counts(normalize=True).round(3).to_frame("share")

,share
rating_sentiment,
positive,0.481
negative,0.370
neutral,0.149


## 3. Complaints: categories, recurring phrases, trend

In [7]:
df = read_feedback_full(get_engine())
analytics.category_breakdown(df).round(2)

,complaint_category,complaints,avg_rating,share
4,network,589,1.55,28.22
3,customer_service,321,1.48,15.38
1,billing,296,1.61,14.18
6,price_value,260,1.62,12.46
2,contract,188,1.51,9.01
0,app_website,179,1.56,8.58
7,sim_activation,154,1.64,7.38
5,other,100,2.00,4.79


In [8]:
analytics.recurring_problems(df, top_n=10)

,phrase,mentions
0,mobile data,152
1,network outage,131
2,data competitors cheaper,111
3,expensive little,111
4,increase worth money,108
5,does work,108
6,hotline solve problem,98
7,called times,98
8,connection extremely slow,94
9,keeps dropping internet,94


In [9]:
analytics.monthly_trend(df).round(2)

,feedback_month,feedback_count,avg_rating,avg_sentiment,complaints,nps
0,2025-01,418,3.15,0.09,174,-24.16
1,2025-02,408,3.22,0.11,160,-22.79
2,2025-03,429,3.27,0.16,150,-20.98
3,2025-04,400,3.34,0.18,144,-13.50
4,2025-05,428,3.24,0.11,175,-22.43
5,2025-06,399,3.34,0.18,139,-15.54
6,2025-07,428,3.21,0.14,171,-17.76
7,2025-08,412,3.26,0.13,161,-20.87
8,2025-09,552,2.72,-0.20,312,-43.84
9,2025-10,428,3.26,0.10,179,-20.79


The injected outage in September shows up as a clear complaint spike:

In [10]:
a = analytics.detect_anomalies(df)
a[a["is_anomaly"]].round(2)

/tmp/ipykernel_1277/4066279365.py:2: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  a[a["is_anomaly"]].round(2)


,period,complaints,expected,spread,z_score,is_anomaly
250,2025-09-08,27,5.0,2.97,7.42,True
251,2025-09-09,35,5.0,2.97,10.12,True
252,2025-09-10,32,5.0,2.97,9.11,True
253,2025-09-11,21,5.0,2.97,5.40,True
254,2025-09-12,32,5.0,2.97,9.11,True
255,2025-09-13,25,5.0,2.97,6.74,True
256,2025-09-14,25,5.5,3.71,5.26,True


## 4. Segments

In [11]:
analytics.segment_table(df, "customer_segment").round(1)

,customer_segment,feedback_count,avg_rating,negative_share,nps
0,at_risk,1053,2.5,60.8,-58.7
1,business,779,3.2,38.9,-25.8
3,new_customer,1289,3.2,37.0,-22.0
2,loyal_customer,2020,3.6,25.7,-3.0


In [12]:
run_query(get_engine(), "topic_overview")[["topic_label", "top_terms", "feedback_count"]]

,topic_label,top_terms,feedback_count
0,app / crashes try / log,"app, crashes try, try log, app crashes, log, try, crashes, does work",829
1,service / support / agent,"service, support, agent, terrible service, hung terrible, support agent, hung, agent hung",352
2,mobile data / connection extremely / dropping internet,"mobile data, mobile, connection extremely, dropping internet, dropping, extremely slow...",152
3,signal / home network / coverage terrible,"signal, home network, home, signal home, network coverage, coverage terrible, terrible...",145
4,outage / calls / drop,"outage, calls, network outage, calls drop, drop, area, constant, constant network",131
5,competitors cheaper / little data / expensive,"competitors cheaper, little data, little, competitors, data competitors, expensive lit...",111
6,increase worth / anymore / money,"increase worth, increase, price increase, anymore, worth money, money anymore, money, ...",108
7,problem / solve / times,"problem, problem called, solve, hotline solve, times, called, called times, solve problem",98
8,network evening / lost connection / today,"network evening, evening, lost connection, lost, connection today, today network, toda...",84
9,joke / speed / 5g,"joke, joke speed, speed, coverage joke, 5g, slower 3g, 5g coverage, speed slower",77


## Findings -> next steps
* Network problems are the #1 complaint driver; the September outage is detectable within a day.
* `at_risk` customers have a strongly negative NPS - a candidate for a churn-prevention analysis.
* Neutral is the hardest sentiment class (lowest F1) - compare with the transformer model
  (`--sentiment-backend transformer`) on a real dataset.

In [13]:
print(analytics.summarize_findings(df))

- **5,141 feedback items** analysed; average rating **3.20/5**, CSAT **48%**, simulated NPS **-23**.
- **38%** of feedback is negative.
- Main complaint drivers: network (28%), customer service (15%), billing (14%).
- Least satisfied segment: **at risk** (NPS -59, 61% negative).
- 7 anomalous day(s) detected; biggest spike on **2025-09-09** with 35 complaints (expected ~5).
- NPS moved **+10.1 points** in 2025-12 vs. the previous month.
